## Decision Tree

"의사결정나무" 로 불리는 머신러닝 학습법을 이해하기 위해서 천천히 구조를 알아보자

### 기본 프로세스

1. 해당 노드에 포함된 샘플이 모두 같은 클래스 일 경우, 더는 분할을 진행하지 않습니다
2. 해당 속성집합이 0 인 경우, 혹은 모든 샘플이 모든 속성에서 같은 값을 취할 경우, 더는 분할을 진행하지 않습니다
3. 해당 노드가 포함하고 있는 샘플의 집합이 0 인 경우, 더는 분할을 진행하지 않습니다

위의 세 가지 상황은 재귀 과정을 일으키는 트리거로 작동합니다

In [1]:
from pandas import read_csv
from sklearn import tree as tree
from scipy import stats as sts
from sklearn import model_selection as mod

data=read_csv("../데이터/고객대출등급예측.csv")

In [2]:
target="대출등급"
feature_col=data.columns.difference(["대출등급","ID"])

In [3]:
train,test=mod.train_test_split(data,stratify=data[target],random_state=1,train_size=0.9)

#### 1. Entropy

"순도"를 나타내는 척도입니다.

$Ent(D)=-\sum_{k=1}^{|y|} p_k \log_2{p_k}$

$p_k$는 집단 $D$의 k번째 클래스의 비율입니다

In [12]:
train[target].value_counts(normalize=True)

대출등급
B    0.299259
C    0.286855
A    0.174178
D    0.138673
E    0.076375
F    0.020297
G    0.004362
Name: proportion, dtype: float64

위의 수치들이 $p_k$를 계산한 결과입니다

In [22]:
pks=train[target].value_counts(normalize=True)
sts.entropy(pk=pks,base=2)

2.3038194310602047

In [18]:
from math import log2

ent=0
for pk in pks:
    ent+=pk*log2(pk)

In [23]:
-ent

2.3038194310602043

물론 $\log_2 x$를 사용하는 것은 $\frac{\log x}{log 2}$ 와 같으므로 로그의 밑은 크게 중요하지 않습니다

### 2. Gain

정보이득은 다음과 같이 계산합니다

이산 속성 $a$가 택할 수 있는 값이 $V$개라 가정합니다

$Gain(D,a)=Ent(D)-\sum_{v=1}^{V} \frac{|D^v|}{|D|} Ent(D^v)$

In [30]:
# 구현해보죠

def ent(D):
    pks=D[target].value_counts(normalize=True)
    return sts.entropy(pk=pks,base=2)

def gain(D,a):
    ent_D=ent(D)
    sub_ent=0
    sub_Ds=[D[D[a]==v] for v in D[a].unique()]

    for sub_D in sub_Ds:
        sub_ent+=len(sub_D)/len(D)*ent(sub_D)

    return ent_D-sub_ent

In [35]:
gain(test,"대출목적")

0.04791224220254353

In [34]:
gain(test,"ID")

2.3035739386358998

### 3. Gain_ratio (C4.5)

위의 "ID"를 기준으로 계산된 정보이득이 굉장히 높다는 것을 알 수 있습니다.

즉, 선택할 수 있는 v의 개수가 많을수록 정보이득이 높아진다는 것을 예상할 수 있습니다.

이 편향이 모델에 악영향을 줄 수 있습니다.

따라서 나온 개념은 정보이득률입니다.

$Gain\_ratio(D,a)=\frac{Gain(D,a)}{IV(a)}$

이 때 IV 값은

$IV(a)=-\sum_{v=1}^{V}\frac{|D^v|}{|D|}log_{2}{\frac{|D^v|}{|D|}}$

즉 IV는 택할 수 있는 속성값이 많을수록 패널티가 커지게 만듭니다

In [40]:
def iv(D,a):
    sub_iv=0
    sub_Ds=[D[D[a]==v] for v in D[a].unique()]

    for sub_D in sub_Ds:
        sub_iv+=log2(len(sub_D)/len(D))*len(sub_D)/len(D)

    return -sub_iv

def gain_ratio(D,a):
    return gain(D,a)/iv(D,a)

In [39]:
iv(test,"ID")

13.233320082732694

In [41]:
iv(test,"대출목적")

1.800545753360122

In [42]:
gain_ratio(test,"ID")

0.17407377167893678

In [43]:
gain_ratio(test,"대출목적")

0.026609844328105082

### 4. Gini (CART)

지니계수는 임의의 두 샘플을 고르고 두 샘플이 다른 클래스에 속할 확률을 나타냅니다

$Gini(D)=\sum_{k=1}{|y|}\sum_{k'\neq k} p_k p_{k'}=1-\sum p_k^2$

당연히 지니계수를 사용할 시 지니계수가 가장 작은 속성을 분할속성으로 택합니다